# Week 4 — Clean and Validate Canadian Business Counts (Niagara)
**Table:** Statistics Canada 33-10-1176-01 (with employees; CMAs and CSDs)
**Task type:** data cleaning only — no EDA, modelling, trend analysis, succession variables or merging with other datasets.

**Principles followed**
- The raw file is opened read-only and its SHA-256 is compared before and after the run.
- Niagara is selected by **DGUID**, never by `GEO` name.
- The CMA is kept in a **separate** file and is never added to municipality totals.
- `REF_DATE` is preserved exactly as published (`2026-01`).
- No zero-value or missing-geography records are created.
- NAICS levels, aggregate categories and employment-size totals are *flagged*, not merged or deleted.

## 0. Setup and paths

In [5]:
import hashlib, json
from pathlib import Path
import pandas as pd

# Locate the repository root (works from repo root or from notebooks/)
ROOT = Path.cwd()
if not (ROOT / "data" / "raw").exists():
    ROOT = ROOT.parent
RAW_PATH = ROOT / "data" / "raw" / "33101176.csv"
INTERIM_DIR  = ROOT / "data" / "interim"        # adjust to the project's agreed cleaned-data folder
REPORT_DIR   = ROOT / "reports"
OUT_MUNI     = INTERIM_DIR / "canadian_business_counts_niagara_cleaned.csv"
OUT_CMA      = INTERIM_DIR / "canadian_business_counts_niagara_cma_reference.csv"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

NAICS_RAW = "North American Industry Classification System (NAICS)"

NIAGARA_MUNICIPALITIES = {
    "2021A00053526003": "Fort Erie",
    "2021A00053526065": "Grimsby",
    "2021A00053526057": "Lincoln",
    "2021A00053526043": "Niagara Falls",
    "2021A00053526047": "Niagara-on-the-Lake",
    "2021A00053526028": "Pelham",
    "2021A00053526011": "Port Colborne",
    "2021A00053526053": "St. Catharines",
    "2021A00053526037": "Thorold",
    "2021A00053526014": "Wainfleet",
    "2021A00053526032": "Welland",
    "2021A00053526021": "West Lincoln",
}
CMA_DGUID = "2021S0503539"      # St. Catharines - Niagara CMA (overlaps the 12 municipalities)

EMPLOYMENT_SIZE_TOTAL = "Total, with employees"
EMPLOYMENT_SIZE_BANDS = [
    "1 to 4 employees", "5 to 9 employees", "10 to 19 employees",
    "20 to 49 employees", "50 to 99 employees", "100 to 199 employees",
    "200 to 499 employees", "500 plus employees",
]
AGGREGATE_NAICS = {
    "1": ("aggregate_total", "Total, all industries"),
    "2": ("unclassified", "Unclassified"),
    "3": ("aggregate_subtotal_classified", "Sub-total, classified"),
}

LOG = {}    # everything reported in the cleaning report is collected here

def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

RAW_HASH_BEFORE = sha256(RAW_PATH)
print("Raw file:", RAW_PATH)
print("SHA-256 before:", RAW_HASH_BEFORE)

Raw file: 33101176.csv
SHA-256 before: 25900b137bf79991719bb4516014317807255338936f59e9ddc1566282f1c8d8


## 1. Load the raw file and run the BEFORE-CLEANING checks
Everything is read as text (`dtype=str`) so nothing is silently re-typed (for example, geography codes with leading zeros). Only truly empty cells become missing; strings such as `"NA"` are not treated as missing.
The task requires an exact match with the Week 3 audit; the cell **stops** if any figure differs.

In [6]:
raw = pd.read_csv(RAW_PATH, dtype=str, keep_default_na=False, na_values=[""], encoding="utf-8-sig")

before = {
    "rows": len(raw),
    "columns": raw.shape[1],
    "ref_dates": sorted(raw["REF_DATE"].unique().tolist()),
    "unique_dguids": int(raw["DGUID"].nunique()),
    "unique_geo_names": int(raw["GEO"].nunique()),
}
print(before)

EXPECTED = {"rows": 1_047_957, "columns": 16, "ref_dates": ["2026-01"], "unique_dguids": 3_348}
for k, v in EXPECTED.items():
    assert before[k] == v, f"STOP - {k}: expected {v}, found {before[k]}. Report to team lead before continuing."
print("All BEFORE-CLEANING checks match the Week 3 audit.")

{'rows': 1047957, 'columns': 16, 'ref_dates': ['2026-01'], 'unique_dguids': 3348, 'unique_geo_names': 3163}
All BEFORE-CLEANING checks match the Week 3 audit.


## 2. Inspect the raw data (before changing anything)

In [7]:
print("Columns:", list(raw.columns))
print()
print("Missing values per column (raw):")
missing_raw = raw.isna().sum()
print(missing_raw.to_string())

before["missing_by_column"] = {k: int(v) for k, v in missing_raw.items()}
before["fully_empty_columns"] = [c for c in raw.columns if raw[c].isna().all()]
before["exact_duplicate_rows"] = int(raw.duplicated().sum())
KEY = ["REF_DATE", "GEO", "DGUID", NAICS_RAW, "Employment size"]
before["key_duplicates"] = int(raw.duplicated(KEY).sum())
print()
print("Fully empty columns:", before["fully_empty_columns"])
print("Exact duplicate rows:", before["exact_duplicate_rows"])
print("Duplicates on REF_DATE+GEO+DGUID+NAICS+Employment size:", before["key_duplicates"])

Columns: ['REF_DATE', 'GEO', 'DGUID', 'Employment size', 'North American Industry Classification System (NAICS)', 'UOM', 'UOM_ID', 'SCALAR_FACTOR', 'SCALAR_ID', 'VECTOR', 'COORDINATE', 'VALUE', 'STATUS', 'SYMBOL', 'TERMINATED', 'DECIMALS']

Missing values per column (raw):
REF_DATE                                                       0
GEO                                                            0
DGUID                                                          0
Employment size                                                0
North American Industry Classification System (NAICS)          0
UOM                                                            0
UOM_ID                                                         0
SCALAR_FACTOR                                                  0
SCALAR_ID                                                      0
VECTOR                                                         0
COORDINATE                                                     0
VALUE      

In [8]:
# Why GEO names cannot be used as the filter: names repeat across Canada
names_multi_dguid = raw.groupby("GEO")["DGUID"].nunique()
before["geo_names_with_multiple_dguids"] = int((names_multi_dguid > 1).sum())
print("GEO names that map to more than one DGUID:", before["geo_names_with_multiple_dguids"])
print("'Lincoln' DGUIDs in the raw file:")
print(raw.loc[raw["GEO"] == "Lincoln", "DGUID"].drop_duplicates().tolist())

GEO names that map to more than one DGUID: 167
'Lincoln' DGUIDs in the raw file:
['2021A00051303008', '2021A00053526057']


In [9]:
# Compare against the Week 3 audit expectations
assert before["fully_empty_columns"] == ["STATUS", "SYMBOL", "TERMINATED"], before["fully_empty_columns"]
assert before["exact_duplicate_rows"] == 0
assert before["key_duplicates"] == 0
for c in ["REF_DATE", "GEO", "DGUID", "Employment size", NAICS_RAW, "VALUE"]:
    assert raw[c].isna().sum() == 0, f"unexpected missing values in {c}"
print("Raw file agrees with the Week 3 audit: STATUS/SYMBOL/TERMINATED are empty; no duplicates; no missing key fields.")

Raw file agrees with the Week 3 audit: STATUS/SYMBOL/TERMINATED are empty; no duplicates; no missing key fields.


## 3. Filter to the 12 Niagara municipalities — by DGUID
`GEO` is **not** used for selection. The CMA (`2021S0503539`) is pulled out separately in the next step.

In [10]:
muni = raw[raw["DGUID"].isin(NIAGARA_MUNICIPALITIES)].copy()

present = muni["DGUID"].drop_duplicates().tolist()
missing_munis = set(NIAGARA_MUNICIPALITIES) - set(present)
assert not missing_munis, f"Missing Niagara municipalities: {missing_munis}"
assert muni["DGUID"].nunique() == 12
assert not muni["DGUID"].eq(CMA_DGUID).any(), "CMA leaked into the municipality dataset"

# Confirm the GEO name in the file agrees with the DGUID we selected (sanity check only)
check = (muni.groupby("DGUID")["GEO"].agg(lambda s: sorted(s.unique())).rename("GEO_in_file").to_frame())
check["expected_name"] = check.index.map(NIAGARA_MUNICIPALITIES)
print(check.to_string())
rows_per_muni = muni.groupby("GEO").size().sort_values(ascending=False)
print()
print(rows_per_muni.to_string())
print("\nMunicipality rows:", len(muni))

                            GEO_in_file        expected_name
DGUID                                                       
2021A00053526003            [Fort Erie]            Fort Erie
2021A00053526011        [Port Colborne]        Port Colborne
2021A00053526014            [Wainfleet]            Wainfleet
2021A00053526021         [West Lincoln]         West Lincoln
2021A00053526028               [Pelham]               Pelham
2021A00053526032              [Welland]              Welland
2021A00053526037              [Thorold]              Thorold
2021A00053526043        [Niagara Falls]        Niagara Falls
2021A00053526047  [Niagara-on-the-Lake]  Niagara-on-the-Lake
2021A00053526053       [St. Catharines]       St. Catharines
2021A00053526057              [Lincoln]              Lincoln
2021A00053526065              [Grimsby]              Grimsby

GEO
St. Catharines         1496
Niagara Falls          1285
Lincoln                 971
Welland                 941
Niagara-on-the-Lake     870
F

## 4. Keep the CMA separate
The CMA geographically **contains** the municipalities, so its counts overlap with theirs. It is saved to its own file for possible later *comparison only* and must never be added to municipality totals.

In [11]:
cma = raw[raw["DGUID"] == CMA_DGUID].copy()
assert len(cma) > 0 and cma["GEO"].nunique() == 1
print("CMA GEO name in file:", cma["GEO"].iloc[0], "| rows:", len(cma))

CMA GEO name in file: St. Catharines - Niagara | rows: 2121


## 5. Reference period
All records carry `REF_DATE = 2026-01`; the Statistics Canada cube title says *June 2026*. The Week 3 audit could not resolve this. The value is **preserved as published**, no corrected date is invented, and no time trends are built from this file.

In [12]:
assert muni["REF_DATE"].unique().tolist() == ["2026-01"]
assert cma["REF_DATE"].unique().tolist() == ["2026-01"]
print("REF_DATE preserved as published:", muni["REF_DATE"].unique().tolist())

REF_DATE preserved as published: ['2026-01']


## 6. Build the cleaned dataset (applied identically to the municipality and CMA subsets)
Steps inside `prepare()`:
1. strip stray whitespace and normalise text (no change in meaning);
2. split `NAICS` into `naics_label` and `naics_code` (**string**);
3. add NAICS hierarchy fields — digit length and level name — without combining levels;
4. flag aggregate / unclassified / normal NAICS categories;
5. flag the employment-size total versus the eight size bands;
6. convert `VALUE` to an integer `business_count` (never fills absent rows);
7. drop technical columns that carry no information (documented below).

`REF_DATE` stays a text value `2026-01`.

In [13]:
import unicodedata

def clean_text(s):
    s = s.astype("string")
    s = s.map(lambda x: unicodedata.normalize("NFC", x) if isinstance(x, str) else x)
    s = (s.str.replace("\u00a0", " ", regex=False)
           .str.replace(r"\s+", " ", regex=True)
           .str.strip())
    return s

def naics_level_name(row_digits, is_aggregate):
    if is_aggregate:
        return "aggregate_not_a_naics_level"
    return {2: "sector", 3: "subsector", 4: "industry_group"}.get(row_digits, "other")

def prepare(frame, geo_level):
    d = frame.copy()
    for c in ["REF_DATE", "GEO", "DGUID", "Employment size", NAICS_RAW]:
        d[c] = clean_text(d[c])

    # ---- NAICS: label + code (code stays TEXT) ----
    ext = d[NAICS_RAW].str.extract(r"^(?P<naics_label>.*?)\s*\[(?P<naics_code>[^\]]+)\]\s*$")
    assert ext["naics_code"].notna().all(), "Some NAICS strings have no [code]"
    d["naics_label"] = ext["naics_label"].astype("string")
    d["naics_code"] = ext["naics_code"].astype("string")

    # ---- NAICS hierarchy (flag only; levels are never combined) ----
    d["naics_is_range"] = d["naics_code"].str.contains("-", regex=False)        # e.g. 31-33
    first_part = d["naics_code"].str.split("-").str[0]
    d["naics_digit_length"] = first_part.str.len().astype("int8")
    d["naics_category_type"] = d["naics_code"].map(lambda c: AGGREGATE_NAICS[c][0] if c in AGGREGATE_NAICS else "industry")
    is_agg = d["naics_category_type"] != "industry"
    d["naics_level_name"] = [naics_level_name(n, a) for n, a in zip(d["naics_digit_length"], is_agg)]

    # ---- Employment size: flag total vs bands (no recoding, no micro/small) ----
    unknown = set(d["Employment size"].unique()) - set([EMPLOYMENT_SIZE_TOTAL] + EMPLOYMENT_SIZE_BANDS)
    assert not unknown, f"Unexpected employment size labels: {unknown}"
    d["employment_size_type"] = d["Employment size"].map(
        lambda x: "aggregate_total" if x == EMPLOYMENT_SIZE_TOTAL else "size_band")

    # ---- VALUE -> integer business_count ----
    v = pd.to_numeric(d["VALUE"], errors="raise")          # raises if anything is non-numeric
    assert v.notna().all() and (v >= 0).all() and (v % 1 == 0).all()
    d["business_count"] = v.astype("int32")

    d["geo_level"] = geo_level

    out = d.rename(columns={
        "REF_DATE": "ref_date", "GEO": "geo", "DGUID": "dguid",
        "Employment size": "employment_size", "VECTOR": "vector",
    })[[
        "ref_date", "dguid", "geo", "geo_level",
        "employment_size", "employment_size_type",
        "naics_code", "naics_label", "naics_digit_length", "naics_level_name",
        "naics_is_range", "naics_category_type",
        "business_count", "vector",
    ]]
    return out.reset_index(drop=True)

muni_clean = prepare(muni, "municipality")
cma_clean  = prepare(cma,  "cma_reference_only")
print(muni_clean.shape, cma_clean.shape)
muni_clean.head(8)

(10462, 14) (2121, 14)


,ref_date,dguid,geo,geo_level,employment_size,employment_size_type,naics_code,naics_label,naics_digit_length,naics_level_name,naics_is_range,naics_category_type,business_count,vector
0,2026-01,2021A00053526065,Grimsby,municipality,"Total, with employees",aggregate_total,1,"Total, all industries",1,aggregate_not_a_naics_level,False,aggregate_total,814,v1918953682
1,2026-01,2021A00053526065,Grimsby,municipality,"Total, with employees",aggregate_total,2,Unclassified,1,aggregate_not_a_naics_level,False,unclassified,52,v1918953696
2,2026-01,2021A00053526065,Grimsby,municipality,"Total, with employees",aggregate_total,3,"Sub-total, classified",1,aggregate_not_a_naics_level,False,aggregate_subtotal_classified,762,v1918953712
3,2026-01,2021A00053526065,Grimsby,municipality,"Total, with employees",aggregate_total,11,"Agriculture, forestry, fishing and hunting",2,sector,False,industry,20,v1918953683
4,2026-01,2021A00053526065,Grimsby,municipality,"Total, with employees",aggregate_total,22,Utilities,2,sector,False,industry,2,v1918953697
5,2026-01,2021A00053526065,Grimsby,municipality,"Total, with employees",aggregate_total,23,Construction,2,sector,False,industry,99,v1918953701
6,2026-01,2021A00053526065,Grimsby,municipality,"Total, with employees",aggregate_total,31-33,Manufacturing,2,sector,True,industry,29,v1918953713
7,2026-01,2021A00053526065,Grimsby,municipality,"Total, with employees",aggregate_total,41,Wholesale trade,2,sector,False,industry,30,v1918953743


### Columns removed from the analytical dataset
Only columns that were either completely empty or constant for every row were removed. They remain in the untouched raw file.
`STATUS`, `SYMBOL` and `TERMINATED` are Statistics Canada **data-quality / footnote** fields — they are **not** business operating status and **not** business closure.

In [14]:
dropped_cols = {}
for c in ["STATUS", "SYMBOL", "TERMINATED", "UOM", "UOM_ID", "SCALAR_FACTOR", "SCALAR_ID", "DECIMALS", "COORDINATE"]:
    vals = muni[c].dropna().unique().tolist()
    if len(vals) == 0:
        dropped_cols[c] = "completely empty in the raw file (not business status / closure information)"
    elif len(vals) == 1 and c != "COORDINATE":
        dropped_cols[c] = f"constant for every row: {vals[0]!r}"
    elif c == "COORDINATE":
        dropped_cols[c] = "positional index of geography/size/NAICS - redundant"
    else:
        raise ValueError(f"{c} has varying values; do not drop without review: {vals[:5]}")
for k, v in dropped_cols.items():
    print(f"{k:14s} -> {v}")

raw_cols_kept = {"REF_DATE", "GEO", "DGUID", "Employment size", NAICS_RAW, "VALUE", "VECTOR"}
assert set(raw.columns) - set(dropped_cols) == raw_cols_kept

STATUS         -> completely empty in the raw file (not business status / closure information)
SYMBOL         -> completely empty in the raw file (not business status / closure information)
TERMINATED     -> completely empty in the raw file (not business status / closure information)
UOM            -> constant for every row: 'Number'
UOM_ID         -> constant for every row: '223'
SCALAR_FACTOR  -> constant for every row: 'units'
SCALAR_ID      -> constant for every row: '0'
DECIMALS       -> constant for every row: '0'
COORDINATE     -> positional index of geography/size/NAICS - redundant


## 7. NAICS validation
- `naics_code` is text.
- Levels are preserved: 1, 2, 3 and 4 digits.
- Parents and children are **not** combined (for example `[61]` and `[611]` both remain).
- `[1]`, `[2]`, `[3]` are flagged as aggregate / unclassified categories, not ordinary industries.

In [15]:
assert str(muni_clean["naics_code"].dtype) == "string"
level_counts = muni_clean.groupby(["naics_digit_length", "naics_level_name", "naics_category_type"]).agg(
    rows=("business_count", "size"), codes=("naics_code", "nunique")).reset_index()
print(level_counts.to_string(index=False))

# Same label at different hierarchy levels is preserved, e.g. Educational services
print()
print(muni_clean.loc[muni_clean["naics_label"] == "Educational services",
                     ["naics_code", "naics_digit_length", "naics_level_name"]].drop_duplicates().to_string(index=False))

 naics_digit_length            naics_level_name           naics_category_type  rows  codes
                  1 aggregate_not_a_naics_level aggregate_subtotal_classified   101      1
                  1 aggregate_not_a_naics_level               aggregate_total   101      1
                  1 aggregate_not_a_naics_level                  unclassified    51      1
                  2                      sector                      industry  1242     20
                  3                   subsector                      industry  3189     95
                  4              industry_group                      industry  5778    285

naics_code  naics_digit_length naics_level_name
        61                   2           sector
       611                   3        subsector


In [16]:
# Aggregate categories are flagged, not deleted
agg_summary = (muni_clean[muni_clean["naics_category_type"] != "industry"]
               .groupby(["naics_code", "naics_label", "naics_category_type"]).size().reset_index(name="rows"))
print(agg_summary.to_string(index=False))

# The 3 aggregate codes must be exactly 1, 2, 3 and no others
assert set(muni_clean.loc[muni_clean["naics_category_type"] != "industry", "naics_code"]) == {"1", "2", "3"}

naics_code           naics_label           naics_category_type  rows
         1 Total, all industries               aggregate_total   101
         2          Unclassified                  unclassified    51
         3 Sub-total, classified aggregate_subtotal_classified   101


## 8. Employment-size validation
Nine source categories are preserved with their original wording. `Total, with employees` is flagged `aggregate_total` so it cannot be added to the eight `size_band` rows. **No micro/small/medium/large grouping is created** — that definition has not been agreed at project level.

In [17]:
sizes = muni_clean.groupby(["employment_size", "employment_size_type"]).size().reset_index(name="rows")
print(sizes.to_string(index=False))
assert muni_clean["employment_size"].nunique() == 9
assert set(muni_clean.loc[muni_clean["employment_size_type"] == "size_band", "employment_size"]) == set(EMPLOYMENT_SIZE_BANDS)
assert (muni_clean.loc[muni_clean["employment_size_type"] == "aggregate_total", "employment_size"] == EMPLOYMENT_SIZE_TOTAL).all()

      employment_size employment_size_type  rows
     1 to 4 employees            size_band  2154
   10 to 19 employees            size_band  1395
 100 to 199 employees            size_band   353
   20 to 49 employees            size_band  1138
 200 to 499 employees            size_band   166
     5 to 9 employees            size_band  1597
   50 to 99 employees            size_band   649
   500 plus employees            size_band    77
Total, with employees      aggregate_total  2933


In [18]:
# Validation only (nothing is saved from this cell): do the published bands reconcile with the published Total?
# Absent rows are NOT treated as zeros in the output data; here we only compare rows that exist.
bands = (muni_clean[muni_clean.employment_size_type == "size_band"]
         .groupby(["dguid", "naics_code"])["business_count"].sum())
tot = (muni_clean[muni_clean.employment_size_type == "aggregate_total"]
       .set_index(["dguid", "naics_code"])["business_count"])
cmp = pd.concat([bands.rename("sum_of_bands"), tot.rename("published_total")], axis=1)
n_cells = len(cmp)
n_total_without_bands = int(cmp["sum_of_bands"].isna().sum())
n_bands_without_total = int(cmp["published_total"].isna().sum())
n_mismatch = int((cmp.dropna()["sum_of_bands"] != cmp.dropna()["published_total"]).sum())
print(f"geography x NAICS cells: {n_cells}; Total with no band rows: {n_total_without_bands}; "
      f"band rows with no Total: {n_bands_without_total}; Total != sum of bands: {n_mismatch}")
LOG["reconciliation"] = dict(cells=n_cells, total_without_bands=n_total_without_bands,
                             bands_without_total=n_bands_without_total, mismatches=n_mismatch)

geography x NAICS cells: 2933; Total with no band rows: 0; band rows with no Total: 0; Total != sum of bands: 0


## 9. Missing-value, duplicate and VALUE validation (after filtering)
Absent geography / NAICS / size combinations are **not** filled with zeros. The source does not let us conclude that a missing combination means zero businesses.

In [19]:
important = ["ref_date", "geo", "dguid", "employment_size", "naics_code", "naics_label", "business_count"]
miss_after = muni_clean.isna().sum()
print(miss_after.to_string())
assert miss_after[important].sum() == 0

# Duplicates: exact + the agreed uniqueness key (DGUID-based; names are not unique)
exact_dups = int(muni_clean.duplicated().sum())
KEY_CLEAN = ["ref_date", "geo", "dguid", "naics_code", "employment_size"]
key_dups = int(muni_clean.duplicated(KEY_CLEAN).sum())
vector_unique = bool(muni_clean["vector"].is_unique)
print("exact duplicates:", exact_dups, "| key duplicates:", key_dups, "| vector unique:", vector_unique)
assert exact_dups == 0 and key_dups == 0 and vector_unique

# VALUE
vs = muni_clean["business_count"]
print(vs.dtype, "min", vs.min(), "max", vs.max(), "| negative:", int((vs < 0).sum()))
assert pd.api.types.is_integer_dtype(vs) and vs.min() >= 0

# Same checks for the CMA reference file
assert cma_clean.isna().sum()[important].sum() == 0
assert cma_clean.duplicated(KEY_CLEAN).sum() == 0

ref_date                0
dguid                   0
geo                     0
geo_level               0
employment_size         0
employment_size_type    0
naics_code              0
naics_label             0
naics_digit_length      0
naics_level_name        0
naics_is_range          0
naics_category_type     0
business_count          0
vector                  0
exact duplicates: 0 | key duplicates: 0 | vector unique: True
int32 min 1 max 4021 | negative: 0


## 10. Final structural checks

In [20]:
assert muni_clean["dguid"].nunique() == 12
assert set(muni_clean["dguid"]) == set(NIAGARA_MUNICIPALITIES)
assert not muni_clean["dguid"].isin([CMA_DGUID]).any()
assert (muni_clean["ref_date"] == "2026-01").all()
assert (muni_clean["geo_level"] == "municipality").all()
assert set(muni_clean.columns).isdisjoint({"STATUS", "SYMBOL", "TERMINATED"})
# Row count must match the filtered raw subset exactly (nothing created, nothing lost)
assert len(muni_clean) == len(muni) and len(cma_clean) == len(cma)

after = {
    "rows": len(muni_clean), "columns": muni_clean.shape[1],
    "municipalities": int(muni_clean["geo"].nunique()),
    "unique_dguids": int(muni_clean["dguid"].nunique()),
    "naics_digit_lengths": sorted(muni_clean["naics_digit_length"].unique().tolist()),
    "naics_codes_unique": int(muni_clean["naics_code"].nunique()),
    "employment_size_categories": muni_clean["employment_size"].unique().tolist(),
    "missing_important": int(miss_after[important].sum()),
    "exact_duplicates": exact_dups, "key_duplicates": key_dups,
    "cma_rows": len(cma_clean),
}
print(json.dumps(after, indent=2))

{
  "rows": 10462,
  "columns": 14,
  "municipalities": 12,
  "unique_dguids": 12,
  "naics_digit_lengths": [
    1,
    2,
    3,
    4
  ],
  "naics_codes_unique": 403,
  "employment_size_categories": [
    "Total, with employees",
    "1 to 4 employees",
    "5 to 9 employees",
    "10 to 19 employees",
    "20 to 49 employees",
    "50 to 99 employees",
    "100 to 199 employees",
    "200 to 499 employees",
    "500 plus employees"
  ],
  "missing_important": 0,
  "exact_duplicates": 0,
  "key_duplicates": 0,
  "cma_rows": 2121
}


## 11. Save the cleaned datasets (new files; the raw file is never written)

In [21]:
muni_clean.to_csv(OUT_MUNI, index=False, encoding="utf-8")
cma_clean.to_csv(OUT_CMA, index=False, encoding="utf-8")

# Re-read to confirm what was written; naics_code must come back as text
chk = pd.read_csv(OUT_MUNI, dtype={"naics_code": "string", "dguid": "string", "ref_date": "string"})
assert chk.shape == muni_clean.shape and chk["dguid"].nunique() == 12
assert chk["ref_date"].unique().tolist() == ["2026-01"]
print("Saved:", OUT_MUNI.name, chk.shape, "|", OUT_CMA.name, cma_clean.shape)

RAW_HASH_AFTER = sha256(RAW_PATH)
assert RAW_HASH_BEFORE == RAW_HASH_AFTER, "RAW FILE CHANGED!"
print("Raw file unchanged. SHA-256:", RAW_HASH_AFTER)

Saved: canadian_business_counts_niagara_cleaned.csv (10462, 14) | canadian_business_counts_niagara_cma_reference.csv (2121, 14)
Raw file unchanged. SHA-256: 25900b137bf79991719bb4516014317807255338936f59e9ddc1566282f1c8d8


## 12. Write the cleaning report

In [22]:
muni_rows = muni_clean.groupby(["geo", "dguid"]).size().reset_index(name="rows").sort_values("geo")
lvl = muni_clean.groupby(["naics_digit_length", "naics_level_name"]).agg(
    rows=("business_count", "size"), codes=("naics_code", "nunique")).reset_index()

def table(df):
    cols = list(df.columns)
    out = ["| " + " | ".join(cols) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    for _, r in df.iterrows():
        out.append("| " + " | ".join(str(r[c]) for c in cols) + " |")
    return "\n".join(out)

dropped_tbl = pd.DataFrame({"raw column": list(dropped_cols), "reason removed": list(dropped_cols.values())})

report = f'''# Cleaning Report - Canadian Business Counts (Niagara), Week 4

**Source:** Statistics Canada table 33-10-1176-01 | **Raw file:** `data/raw/33101176.csv` (unchanged; SHA-256 `{RAW_HASH_AFTER}`)
**Notebook:** `scripts/02_cbc_cleaning.ipynb` | **Outputs:** `{OUT_MUNI.name}`, `{OUT_CMA.name}`

## 1. Before cleaning
| Check | Result | Week 3 audit |
|---|---|---|
| Rows | {before["rows"]:,} | 1,047,957 |
| Columns | {before["columns"]} | 16 |
| REF_DATE values | {", ".join(before["ref_dates"])} | 2026-01 |
| Unique DGUIDs | {before["unique_dguids"]:,} | 3,348 |
| Unique GEO names | {before["unique_geo_names"]:,} | - |
| GEO names mapping to >1 DGUID | {before["geo_names_with_multiple_dguids"]} | names are not unique |
| Exact duplicate rows | {before["exact_duplicate_rows"]} | 0 |
| Duplicates on REF_DATE+GEO+DGUID+NAICS+Employment size | {before["key_duplicates"]} | 0 |
| Fully empty columns | {", ".join(before["fully_empty_columns"])} | STATUS, SYMBOL, TERMINATED |
| Missing values in REF_DATE, GEO, DGUID, Employment size, NAICS, VALUE | 0 | 0 |

**Geographic coverage:** all of Canada ({before["unique_dguids"]:,} geographies: census metropolitan areas and census subdivisions).

## 2. Cleaning actions
1. **Niagara filtering - by DGUID.** Rows were selected with the 12 municipality DGUIDs supplied in the task. `GEO` was not used, because {before["geo_names_with_multiple_dguids"]} names occur under more than one DGUID (e.g. another *Lincoln* exists outside Niagara). All 12 municipalities were confirmed present.
2. **DGUID treatment.** DGUID is the geography key. The `GEO` name was kept only as a readable label and was checked against the expected name for each DGUID.
3. **CMA treatment.** The St. Catharines - Niagara CMA (`{CMA_DGUID}`) overlaps the municipalities. It is saved in a separate file (`{OUT_CMA.name}`, {len(cma_clean):,} rows, `geo_level = cma_reference_only`) and **must never be added to municipality totals**.
4. **NAICS extraction.** `NAICS` was split into `naics_label` and `naics_code`. `naics_code` is stored as text.
5. **NAICS hierarchy.** `naics_digit_length` and `naics_level_name` identify the level (1, 2, 3 or 4 digits). `naics_is_range` marks range codes such as `31-33` (treated as 2-digit sectors). Parent and child categories are **not** combined; for example Educational services `[61]` and `[611]` both remain.
6. **Aggregate categories.** `naics_category_type` flags `[1]` Total, all industries (`aggregate_total`), `[2]` Unclassified (`unclassified`) and `[3]` Sub-total, classified (`aggregate_subtotal_classified`). Every other code is `industry`. Nothing was deleted; filter on this column before analysing industries.
7. **Employment size.** All nine source categories are retained with original wording. `employment_size_type` marks `Total, with employees` as `aggregate_total` and the eight others as `size_band`. No micro/small/medium/large recoding was applied.
8. **Reference period.** `REF_DATE` is preserved as published (`2026-01`). It was not changed to June 2026.
9. **VALUE.** Converted to integer `business_count`; no missing, negative or non-numeric values. No absent combination was filled with zero.
10. **Renamed columns:** REF_DATE->ref_date, GEO->geo, DGUID->dguid, Employment size->employment_size, VECTOR->vector, VALUE->business_count; NAICS -> naics_label + naics_code.
11. **Columns removed from the analytical dataset** (they remain in the raw file):

{table(dropped_tbl)}

## 3. After cleaning
| Item | Result |
|---|---|
| Final rows (12 municipalities) | {after["rows"]:,} |
| Final columns | {after["columns"]} |
| Municipalities | {after["municipalities"]} |
| Unique DGUIDs | {after["unique_dguids"]} |
| NAICS digit lengths retained | {", ".join(str(x) for x in after["naics_digit_lengths"])} |
| Distinct NAICS codes | {after["naics_codes_unique"]} |
| Employment-size categories retained | {len(after["employment_size_categories"])} |
| Missing values in analytical fields | {after["missing_important"]} |
| Exact duplicate rows | {after["exact_duplicates"]} |
| Duplicates on REF_DATE+GEO+DGUID+NAICS+Employment size | {after["key_duplicates"]} |
| CMA reference rows (separate file) | {after["cma_rows"]:,} |

**Rows per municipality**

{table(muni_rows)}

**NAICS levels retained**

{table(lvl)}

**Employment-size categories retained:** {"; ".join(after["employment_size_categories"])}.

**Reconciliation check (validation only):** across {LOG["reconciliation"]["cells"]:,} municipality x NAICS cells, {LOG["reconciliation"]["mismatches"]} had a published Total that differed from the sum of its published bands; {LOG["reconciliation"]["total_without_bands"]} had a Total with no band rows and {LOG["reconciliation"]["bands_without_total"]} had band rows with no Total. Totals were not added to bands anywhere in the output.

## 4. Remaining limitations
- **Single reference period.** All rows are `2026-01`; the file cannot show trends and must not be used for business-count trends on its own.
- **Reference-period discrepancy.** The cube title says June 2026 but the data say 2026-01. The Week 3 audit could not resolve this; the raw value was preserved and no date was invented.
- **Employer businesses only.** The table counts businesses *with employees*; businesses without employees are not represented.
- **No direct succession variables.** There is no owner age, retirement intention, successor, ownership transfer, closure reason or opening year. Nothing of this kind was created or inferred.
- **Aggregate data.** Rows are counts of businesses per geography x NAICS x employment-size cell, not individual businesses.
- **Overlapping hierarchy and categories.** NAICS levels are nested, aggregate NAICS rows overlap detailed ones, and `Total, with employees` overlaps the size bands. Filter on `naics_level_name`, `naics_category_type` and `employment_size_type` before summing, or businesses will be double counted.
- **Absent rows are not zeros.** A missing geography / NAICS / size combination cannot be assumed to mean zero businesses.
- **CMA overlaps the municipalities.** Do not add the CMA file to municipality totals.
- **STATUS, SYMBOL and TERMINATED** were empty in the raw file. They are not business status or closure information.
- **Size definitions.** Micro/small/medium/large are not defined by the source; any grouping requires a project-level definition documented before use.
'''
(REPORT_DIR / "cbc_cleaning_report.md").write_text(report, encoding="utf-8")
print("Report written:", REPORT_DIR / "cbc_cleaning_report.md")

Report written: c:\Users\SITOG-External Sales\Desktop\week 4\reports\cbc_cleaning_report.md
